In [6]:
"""
26 Feb 2026
phase_portraits.py
==================
Phase portraits for the extended Abrams-Strogatz model with demographic growth.

Reconstructed from language_dynamics_extended_v5.py (23 Feb 2026) and updated
to the three-scenario architecture used in all other scripts in this repository.

THREE SCENARIOS
---------------
Configuration A  — single global fit, 1895-2020  (a=1.25)
Configuration B  — dual-regime:
  B1: Consensus   1895-1970  (a=1.478, analytically derived New Consensus params)
  B2: Coexistence 1970-2020  (a=0.655)

For each scenario and each snapshot time the script:
  1. Selects the correct (s_o, s_l, a) — regime-aware for scenario A/B
  2. Computes the (dx_o/dt, dI/dt) vector field on a 2D grid
  3. Finds the dx_o/dt = 0 nullcline
  4. Locates the stable fixed point and computes both eigenvalues
     λ₁ = J₀₀  (x_o direction)
     λ₂ = J₁₁  (I direction)
  5. Classifies: SADDLE (λ₂ > 0) or STABLE NODE (λ₂ < 0)

run_vector_field_analysis(scenario, time_points, nx, nI, I_margin)
  → generates one figure per time point and returns a list of (t_star, fig).

main() generates portraits for both scenarios at the same time points and
saves them as phase_portrait_{scenario}_t{t}.jpg.

Author: Riccardo Del Gratta
Date: February 2026
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from scipy.optimize import brentq, fsolve
import os
import warnings

# ============================================================================
# MODEL PARAMETERS  (identical across all scripts in this repository)
# ============================================================================

N0_indigenous = 1012848
K_indigenous  = 12918933
r_indigenous  = 0.022

N0_spanish = 3117878
K_spanish  = 165995301
r_spanish  = 0.037

A_p   = 5.47
nu    = 0.271
p_max = 0.97

base_year = 1895

# ---------------------------------------------------------------------------
# THREE SCENARIOS
# ---------------------------------------------------------------------------
S_O_CONS_ALL = 0.0349;  S_L_CONS_ALL = 0.0055;  A_CONS_ALL = 1.25    # A
S_O_CONS     = 0.0454;  S_L_CONS     = 0.00839; A_CONS     = 1.478   # B1
S_O_COEX     = 0.047;   S_L_COEX     = 0.015;   A_COEX     = 0.6553  # B2

T_REGIME_CHANGE = 75.0   # t = year 1970

# Year when I crosses K_I/2 (J₁₁ sign change: saddle → stable node)
_T_CROSS = brentq(
    lambda t: K_indigenous / (1 + ((K_indigenous - N0_indigenous) / N0_indigenous)
                              * np.exp(-r_indigenous * t)) - K_indigenous / 2,
    0, 300
)
YEAR_CROSS_I = base_year + _T_CROSS   # ≈ 2007

SCENARIOS = {
    'A': {
        'label':    'A (a=1.25, single-fit 1895-2020)',
        'color':    'steelblue',
        'cmap':     'Blues',
        'fp_color': 'steelblue',
        'pre':  dict(s_o=S_O_CONS_ALL, s_l=S_L_CONS_ALL, a=A_CONS_ALL),
        'post': dict(s_o=S_O_CONS_ALL, s_l=S_L_CONS_ALL, a=A_CONS_ALL),
    },
    'B': {
        'label':    'B (B1: a=1.478 / B2: a=0.655)',
        'color':    'tomato',
        'cmap':     'Reds',
        'fp_color': 'tomato',
        'pre':  dict(s_o=S_O_CONS,  s_l=S_L_CONS,  a=A_CONS),
        'post': dict(s_o=S_O_COEX,  s_l=S_L_COEX,  a=A_COEX),
    },
}

# Census data — bilingual fraction x_o_obs = Bilingual / Indigenous
CENSUS_DATA = {
    1895: 0.2886, 1900: 0.2949, 1910: 0.2951, 1930: 0.4754,
    1940: 0.5032, 1950: 0.6752, 1960: 0.6353, 1970: 0.7235,
    1980: 0.7591, 1990: 0.8352, 1995: 0.8519, 2000: 0.8310,
    2005: 0.8772, 2010: 0.8479, 2020: 0.8895,
}


# ============================================================================
# SCENARIO SELECTOR
# ============================================================================

def scenario_params(t, scenario='A'):
    """
    Return (s_o, s_l, a, sub_label) for the given scenario at time t.

    For scenario A the parameters are constant (single fit).
    For scenario B the parameters switch at T_REGIME_CHANGE (year 1970).
    """
    sc = SCENARIOS[scenario]
    if t < T_REGIME_CHANGE:
        p   = sc['pre']
        lbl = ('A (1895-1970, a=1.25)'    if scenario == 'A'
               else 'B1 — Consensus (a=1.478, 1895-1970)')
    else:
        p   = sc['post']
        lbl = ('A (1970-2020, a=1.25)'    if scenario == 'A'
               else 'B2 — Coexistence (a=0.655, 1970-2020)')
    return p['s_o'], p['s_l'], p['a'], lbl


# ============================================================================
# CORE FUNCTIONS  (self-contained copy)
# ============================================================================

def logistic(t, K, r, N0):
    return K / (1 + ((K - N0) / N0) * np.exp(-r * t))

def m_si(S, I):
    return S / I if I > 0 else np.inf

def p_o_func(m):
    return p_max / (1 + A_p * np.exp(-nu * m))

def f_func(x_o, s_o, s_l, a):
    if x_o <= 0 or x_o >= 1:
        return 0.0
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return s_o * (x_o**a) * (1 - x_o) - s_l * ((1 - x_o)**a) * x_o

def g_func(x_o, p, m, I):
    bracket = p - x_o - p * (1 - p/p_max) * nu * m
    return bracket * r_indigenous * (1 - I/K_indigenous)

def dxo_dt(x_o, I, S, s_o, s_l, a):
    """Full dx_o/dt = f + g  (S frozen at snapshot value)."""
    m = m_si(S, I)
    p = p_o_func(m)
    return f_func(x_o, s_o, s_l, a) + g_func(x_o, p, m, I)

def dI_dt(I):
    """Logistic dI/dt."""
    return r_indigenous * I * (1 - I / K_indigenous)


# ============================================================================
# FIXED-POINT FINDER & JACOBIAN
# ============================================================================

def find_fixed_points(t_star, S_t, s_o, s_l, a, n_scan=3000):
    """
    Find all zeros of dx_o/dt = 0 for fixed I(t_star), S_t, (s_o, s_l, a).

    Returns list of (x_o, I, S, p, s_o, s_l, a) tuples.
    """
    I_t = logistic(t_star, K_indigenous, r_indigenous, N0_indigenous)
    m_t = m_si(S_t, I_t)
    p_t = p_o_func(m_t)

    xs   = np.linspace(0.01, 0.99, n_scan)
    vals = np.array([dxo_dt(x, I_t, S_t, s_o, s_l, a) for x in xs])
    scs  = np.where(np.diff(np.sign(vals)))[0]

    roots = []
    for idx in scs:
        try:
            xr = brentq(lambda x: dxo_dt(x, I_t, S_t, s_o, s_l, a),
                        xs[idx], xs[idx+1], xtol=1e-10)
            roots.append((xr, I_t, S_t, p_t, s_o, s_l, a))
        except Exception:
            mid = 0.5 * (xs[idx] + xs[idx+1])
            sol = fsolve(lambda x: dxo_dt(x[0], I_t, S_t, s_o, s_l, a),
                         [mid], full_output=True)
            if sol[2] == 1 and 0 < sol[0][0] < 1:
                roots.append((sol[0][0], I_t, S_t, p_t, s_o, s_l, a))
    return roots


def calculate_jacobian(x_o, I, S, p, s_o, s_l, a):
    """
    2×2 Jacobian of (dx_o/dt, dI/dt).

    Upper-triangular: eigenvalues are J₀₀ (x_o direction) and J₁₁ (I direction).
      λ₁ = J₀₀ = d(dx_o/dt)/dx_o  — computed by centred finite differences
      λ₂ = J₁₁ = r_I(1 - 2I/K_I)  — analytic, changes sign at I = K_I/2
    """
    eps = 1e-6
    xp  = min(x_o + eps, 0.999)
    xm  = max(x_o - eps, 0.001)
    J00 = (dxo_dt(xp, I, S, s_o, s_l, a)
          -dxo_dt(xm, I, S, s_o, s_l, a)) / (xp - xm)

    h   = p * (1 - nu * (S/I)) + (nu/p_max) * (S/I) * p**2 - x_o
    J01 = (-h * (r_indigenous/K_indigenous)
           + r_indigenous * (1 - I/K_indigenous)
           * nu**2 * (S**2/I**3) * p * (1 - p/p_max) * (1 - 2*p/p_max))
    J11 = r_indigenous * (1 - 2 * I / K_indigenous)

    return np.array([[J00, J01], [0.0, J11]])


def analyze_stability(fixed_points):
    """
    Classify each fixed point from find_fixed_points().

    Returns list of dicts with:
      x_o, I, S, p, s_o, s_l, a
      J00, J11        — diagonal Jacobian entries (= eigenvalues λ₁, λ₂)
      fully_stable    — bool (both eigenvalues < 0)
      saddle          — bool (one < 0, one > 0)
      tau_x           — relaxation time 1/|J00|
    """
    results = []
    for x_o, I, S, p, s_o, s_l, a in fixed_points:
        J   = calculate_jacobian(x_o, I, S, p, s_o, s_l, a)
        J00 = J[0, 0]
        J11 = J[1, 1]

        fully_stable = (J00 < 0) and (J11 < 0)
        saddle       = (J00 < 0) and (J11 > 0)
        tau_x        = 1.0 / abs(J00) if J00 < 0 else np.inf

        results.append(dict(
            x_o=x_o, I=I, S=S, p=p,
            s_o=s_o, s_l=s_l, a=a,
            J00=J00, J11=J11,
            fully_stable=fully_stable, saddle=saddle,
            tau_x=tau_x,
        ))
    return results


# ============================================================================
# VECTOR FIELD
# ============================================================================

def compute_vector_field(t_star, s_o, s_l, a, nx=28, nI=28, I_margin=0.6):
    """
    Compute (dx_o/dt, dI/dt) on a 2D grid  [x_o × I].

    S is frozen at S(t_star) (quasi-static approximation).
    Returns a dict with all arrays needed for plotting.
    """
    I_t = logistic(t_star, K_indigenous, r_indigenous, N0_indigenous)
    S_t = logistic(t_star, K_spanish,    r_spanish,    N0_spanish)
    m_t = m_si(S_t, I_t)
    p_t = p_o_func(m_t)

    x_arr = np.linspace(0.02, 0.98, nx)
    I_lo  = max(I_t * (1 - I_margin), 0.01 * K_indigenous)
    I_hi  = min(I_t * (1 + I_margin), 0.999 * K_indigenous)
    I_arr = np.linspace(I_lo, I_hi, nI)

    X, IG = np.meshgrid(x_arr, I_arr)
    U = np.zeros_like(X)
    V = np.zeros_like(X)

    for i in range(nI):
        for j in range(nx):
            U[i, j] = dxo_dt(X[i, j], IG[i, j], S_t, s_o, s_l, a)
            V[i, j] = dI_dt(IG[i, j])

    magnitude = np.sqrt(U**2 + V**2)
    mag95     = np.percentile(magnitude[magnitude > 0], 95) if np.any(magnitude > 0) else 1.0
    U_norm    = np.where(magnitude > 0, U / mag95, 0)
    V_norm    = np.where(magnitude > 0, V / mag95, 0)

    return dict(X=X, IG=IG, U=U, V=V, U_norm=U_norm, V_norm=V_norm,
                magnitude=magnitude,
                S_t=S_t, I_t=I_t, p_t=p_t, m_t=m_t,
                x_arr=x_arr, I_arr=I_arr)


def compute_nullcline(t_star, I_arr, s_o, s_l, a, n_scan=2000):
    """
    dx_o/dt = 0 nullcline: for each I in I_arr find x_o where rhs = 0.

    S is frozen at S(t_star).
    """
    S_t = logistic(t_star, K_spanish, r_spanish, N0_spanish)
    nullcline = []

    for I_val in I_arr:
        if I_val <= 0:
            continue
        m_val = m_si(S_t, I_val)
        p_val = p_o_func(m_val)

        xs   = np.linspace(0.02, 0.98, n_scan)
        vals = np.array([dxo_dt(x, I_val, S_t, s_o, s_l, a) for x in xs])
        scs  = np.where(np.diff(np.sign(vals)))[0]

        for idx in scs:
            try:
                xn = brentq(lambda x: dxo_dt(x, I_val, S_t, s_o, s_l, a),
                            xs[idx], xs[idx+1], xtol=1e-8)
                nullcline.append((xn, I_val))
            except Exception:
                pass

    return nullcline


# ============================================================================
# SINGLE PORTRAIT
# ============================================================================

def plot_phase_portrait(t_star, scenario='A', nx=28, nI=28, I_margin=0.6):
    """
    Generate one phase portrait for the given scenario at snapshot t_star.

    Displays:
      - Streamplot coloured by vector magnitude
      - dx_o/dt = 0 nullcline (red)
      - I = K_I/2 reference line (grey dashed — J₁₁ sign change ~2007)
      - I(t_star) reference line (orange dotted)
      - Stable fixed point with λ₁ (J₀₀) and λ₂ (J₁₁) annotations
      - Census x_o_obs line if the year matches a census year
      - Parameter info box (top-left)

    Returns matplotlib Figure.
    """
    year    = int(base_year + t_star)
    s_o, s_l, a, sub_label = scenario_params(t_star, scenario)
    sc = SCENARIOS[scenario]

    # ── compute ───────────────────────────────────────────────────────────
    vf       = compute_vector_field(t_star, s_o, s_l, a, nx=nx, nI=nI, I_margin=I_margin)
    nullcl   = compute_nullcline(t_star, vf['I_arr'], s_o, s_l, a)
    fps      = find_fixed_points(t_star, vf['S_t'], s_o, s_l, a)
    sta      = analyze_stability(fps)

    # ── figure ────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, 9))

    # Streamplot
    mag  = vf['magnitude']
    norm = mcolors.Normalize(vmin=0, vmax=np.percentile(mag[mag > 0], 95))
    ax.streamplot(vf['X'], vf['IG'], vf['U_norm'], vf['V_norm'],
                  density=1.2, color=mag, cmap=sc['cmap'],
                  norm=norm, linewidth=0.8, arrowsize=1.0)
    sm = cm.ScalarMappable(cmap=sc['cmap'], norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, label='Vector magnitude (normalised)')

    # Nullcline dx_o/dt = 0
    if nullcl:
        nc_x, nc_I = zip(*sorted(nullcl, key=lambda p: p[1]))
        ax.plot(nc_x, nc_I, color='red', lw=2.0, zorder=4,
                label=r'$\dot{x}_o = 0$ nullcline')

    # I = K_I/2 reference (J₁₁ sign change)
    ax.axhline(K_indigenous / 2, color='grey', lw=1.2, ls='--',
               label=f'$I = K_I/2$  (saddle→stable node, year {YEAR_CROSS_I:.0f})',
               zorder=3)

    # I(t_star) reference
    ax.axhline(vf['I_t'], color='darkorange', lw=1.0, ls=':',
               label=fr'$I(\bar{{t}}) = {vf["I_t"]:.0f}$', zorder=3)

    # Fixed points
    for d in sta:
        if d['fully_stable']:
            clr, mkr, fp_lbl = 'green', 'o', 'Stable node'
        elif d['saddle']:
            clr, mkr, fp_lbl = sc['fp_color'], 's', 'Saddle'
        else:
            clr, mkr, fp_lbl = 'red', '^', 'Unstable'

        ax.scatter(d['x_o'], d['I'], color=clr, marker=mkr,
                   s=150, zorder=7, edgecolors='k', lw=0.9,
                   label=f"{fp_lbl}  ($x_o^*={d['x_o']:.4f}$)")

        # Annotate λ₁ and λ₂ + I value
        s1 = '+' if d['J00'] >= 0 else ''
        s2 = '+' if d['J11'] >= 0 else ''
        txt = (fr"$\lambda_1={s1}{d['J00']:.4f}$"
               "\n"
               fr"$\lambda_2={s2}{d['J11']:.4f}$"
               "\n"
               fr"$I={d['I']:,.0f}$")
        ax.annotate(txt,
                    xy=(d['x_o'], d['I']),
                    xytext=(d['x_o'] + 0.045, d['I']),
                    fontsize=7.5, color='black',
                    arrowprops=dict(arrowstyle='-', color='grey', lw=0.5))

    # Census line if year matches
    if year in CENSUS_DATA:
        xo_obs = CENSUS_DATA[year]
        ax.axvline(xo_obs, color='darkorange', lw=1.5, ls='-.',
                   label=fr'Census $x_o^{{\rm obs}} = {xo_obs:.3f}$ ({year})',
                   zorder=5)

    # Parameter info box (top-left)
    m_bar = vf['m_t']
    p_bar = vf['p_t']
    param_txt = (f'$s_o={s_o}$, $s_l={s_l}$, $a={a}$\n'
                 f'$p_{{\\max}}={p_max}$, $\\nu={nu}$, $A={A_p}$\n'
                 f'$m_{{si}}={m_bar:.2f}$, $p_o={p_bar:.4f}$')
    ax.text(0.02, 0.97, param_txt, transform=ax.transAxes,
            fontsize=8.5, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.88))

    # Regime shading
    if t_star < T_REGIME_CHANGE:
        ax.set_facecolor('#f0f4ff')
        regime_note = 'Consensus regime'
    else:
        ax.set_facecolor('#fff4f0')
        regime_note = 'Coexistence regime'

    # Labels & title
    ax.set_xlabel(r'$x_o$  (bilingual fraction of indigenous population)', fontsize=12)
    ax.set_ylabel(r'$I$  (indigenous population)', fontsize=12)
    ax.set_title(
        f'Phase portrait — Configuration {scenario} — {sub_label}\n'
        f'$\\bar{{t}} = {t_star:.0f}$  (year {year})  [{regime_note}]\n'
        f'$S = {vf["S_t"]:.0f}$,  $m_{{si}} = {m_bar:.2f}$,  '
        f'$p_o = {p_bar:.3f}$',
        fontsize=11)
    ax.set_xlim(vf['x_arr'].min(), vf['x_arr'].max())
    ax.set_ylim(vf['I_arr'].min(), vf['I_arr'].max())
    ax.grid(alpha=0.22)

    # Deduplicate legend entries
    handles, labels = ax.get_legend_handles_labels()
    seen = {}
    for h, l in zip(handles, labels):
        if l not in seen:
            seen[l] = h
    ax.legend(seen.values(), seen.keys(), fontsize=9,
              loc='upper center', bbox_to_anchor=(0.5, -0.11),
              ncol=3, borderaxespad=0, frameon=True)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.22)
    
    return fig


# ============================================================================
# BATCH RUNNER
# ============================================================================

def run_vector_field_analysis(scenario, time_points,
                               nx=28, nI=28, I_margin=0.6,
                               plot_dir='plots/phase_portraits/'):
    """
    Generate one phase portrait per time point for the given scenario.

    Parameters
    ----------
    scenario    : str   'A' or 'B'
    time_points : list  times in years since base_year (e.g. [45, 75, 95, 112])
    nx, nI      : int   grid resolution
    I_margin    : float vertical extent (fraction of I(t_star))
    plot_dir    : str   output directory

    Returns
    -------
    list of (t_star, fig) tuples
    """
    sc   = SCENARIOS[scenario]
    W    = 72
    print("\n" + "=" * W)
    print(f"  PHASE PORTRAITS — Configuration {scenario}: {sc['label']}")
    print(f"  Grid: {nx}×{nI},  I_margin={I_margin}")
    print(f"  J₁₁ sign change (saddle→stable) at year {YEAR_CROSS_I:.0f}")
    print("=" * W)

    os.makedirs(plot_dir, exist_ok=True)
    figures = []

    for t_star in time_points:
        year = int(base_year + t_star)
        s_o, s_l, a, sub_label = scenario_params(t_star, scenario)
        print(f"\n  t={t_star:.0f}  (year {year})  [{sub_label}] ...", end='', flush=True)

        fps = find_fixed_points(t_star,
                                logistic(t_star, K_spanish, r_spanish, N0_spanish),
                                s_o, s_l, a)
        sta = analyze_stability(fps)
        fig = plot_phase_portrait(t_star, scenario=scenario,
                                  nx=nx, nI=nI, I_margin=I_margin)
        fname = f"{plot_dir}phase_portrait_{scenario}_t{int(t_star)}.jpg"
        fig.savefig(fname, dpi=300, bbox_inches='tight')
        figures.append((t_star, fig))
        print(f" saved → {fname}")

        # Print stability summary for this snapshot
        for d in sta:
            regime = ("SADDLE"      if d['saddle']       else
                      "STABLE NODE" if d['fully_stable']  else
                      "UNSTABLE")
            print(f"    x_o* = {d['x_o']:.5f}   I = {d['I']:.0f}   "
                  f"λ₁(J₀₀) = {d['J00']:+.5f}   "
                  f"λ₂(J₁₁) = {d['J11']:+.5f}   [{regime}]")

    return figures


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":

    import matplotlib
    matplotlib.use('Agg')

    plot_dir = 'plots/phase_portraits/'
    os.makedirs(plot_dir, exist_ok=True)

    # Time points that span pre/post-transition and include the census saddles.
    # Pre-1970 portraits (t < 75): consensus regime (B → uses B1 params)
    # Post-1970 portraits (t ≥ 75): coexistence regime (B → uses B2 params)
    # t=112 ≈ year 2007: I ≈ K_I/2, saddle→stable transition
    time_points_portraits = [
        45.0,   # year 1940 — deep consensus/single-fit, saddle
        65.0,   # year 1960 — approaching regime change, saddle
        75.0,   # year 1970 — regime change (B: B1→B2)
        85.0,   # year 1980 — post-regime, saddle
        95.0,   # year 1990 — approaching J₁₁ = 0, saddle
       105.0,   # year 2000 — near transition, saddle
       112.0,   # year 2007 — I ≈ K_I/2, J₁₁ ≈ 0
       115.0,   # year 2010 — stable node
       122.0,   # year 2017 — stable node
    ]

    print("\n" + "#"*72)
    print("# phase_portraits.py")
    print("# Extended Abrams-Strogatz model — Phase portraits")
    print("# Configurations A, B  (three-scenario architecture)")
    print(f"# J₁₁ sign change at year {YEAR_CROSS_I:.0f}")
    print("#"*72)

    all_figs = {}
    for sc in ('A', 'B'):
        figs = run_vector_field_analysis(
            sc, time_points_portraits,
            nx=28, nI=28, I_margin=0.6,
            plot_dir=plot_dir
        )
        all_figs[sc] = figs
        plt.close('all')

    # ── Summary table ──────────────────────────────────────────────────────
    print("\n" + "="*85)
    print("  STABILITY SUMMARY — both scenarios at all time points")
    print(f"  {'Year':>4}  {'Scen':>4}  {'Sub-regime':>30}  "
          f"{'x_o*':>7}  {'λ₁(J₀₀)':>10}  {'λ₂(J₁₁)':>10}  {'Class':>12}")
    print("  " + "-"*82)

    for t_star in time_points_portraits:
        year = int(base_year + t_star)
        for sc in ('A', 'B'):
            s_o, s_l, a, sub_lbl = scenario_params(t_star, sc)
            S_t = logistic(t_star, K_spanish, r_spanish, N0_spanish)
            fps = find_fixed_points(t_star, S_t, s_o, s_l, a)
            sta = analyze_stability(fps)
            for d in sta:
                cls = ("Saddle"      if d['saddle']       else
                       "Stable node" if d['fully_stable']  else
                       "Unstable")
                print(f"  {year:>4}  {sc:>4}  {sub_lbl:>30}  "
                      f"{d['x_o']:>7.4f}  {d['J00']:>+10.5f}  "
                      f"{d['J11']:>+10.5f}  {cls:>12}")
        print()

    print(f"\nAll figures saved to {plot_dir}")
    print("#"*72)


########################################################################
# phase_portraits.py
# Extended Abrams-Strogatz model — Phase portraits
# Configurations A, B  (three-scenario architecture)
# J₁₁ sign change at year 2007
########################################################################

  PHASE PORTRAITS — Configuration A: A (a=1.25, single-fit 1895-2020)
  Grid: 28×28,  I_margin=0.6
  J₁₁ sign change (saddle→stable) at year 2007

  t=45  (year 1940)  [A (1895-1970, a=1.25)] ... saved → plots/phase_portraits/phase_portrait_A_t45.jpg
    x_o* = 0.35363   I = 2406699   λ₁(J₀₀) = -0.00668   λ₂(J₁₁) = +0.01380   [SADDLE]

  t=65  (year 1960)  [A (1895-1970, a=1.25)] ... saved → plots/phase_portraits/phase_portrait_A_t65.jpg
    x_o* = 0.50060   I = 3388048   λ₁(J₀₀) = -0.01202   λ₂(J₁₁) = +0.01046   [SADDLE]

  t=75  (year 1970)  [A (1970-2020, a=1.25)] ... saved → plots/phase_portraits/phase_portrait_A_t75.jpg
    x_o* = 0.60075   I = 3965834   λ₁(J₀₀) = -0.01683   λ₂(J₁₁)